### Import des données

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(123)

In [12]:
%pip install s3fs
import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "mariedominge"
print(fs.ls(MY_BUCKET))


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
['mariedominge/donnees_GBD_uniques.csv']


In [13]:
with fs.open(f"{MY_BUCKET}/donnees_GBD_uniques.csv", "rb") as f:
    df_GBD = pd.read_csv(f, sep=";", encoding="latin-1", decimal=",")

df_GBD.head()

,population_group,measure,location_id,location_name,sex,age_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
0,All Population,Deaths,71,Australia,Both,All ages,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600
1,All Population,Deaths,71,Australia,Both,All ages,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100
2,All Population,Deaths,71,Australia,Both,All ages,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387
3,All Population,Deaths,71,Australia,Both,All ages,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975
4,All Population,Deaths,71,Australia,Both,All ages,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420


### Modification du dataframe GBD pour faire correspondre les causes_GBD avec les causes_2006 (on passe de 14 causes à 9 causes)

In [14]:
# création d'un dataframe qui ne comporte que les infos utiles == qui different d'un pays à l'autre, etc 

df_GBD_modifie = df_GBD[["location_id", "location_name", "cause_id", "cause_name", "metric_id", "metric_name", "year", "value","upper_bound", "lower_bound"]]

# conversion du texte en valeurs pour value, upper_bound, lower_bound
df_GBD_modifie["value"] = pd.to_numeric(df_GBD_modifie["value"])
df_GBD_modifie["upper_bound"] = pd.to_numeric(df_GBD_modifie["upper_bound"])
df_GBD_modifie["lower_bound"] = pd.to_numeric(df_GBD_modifie["lower_bound"])

# ajout des écarts-types et variances pour après calculer les nouvelles upper_bound et lower_bound

df_GBD_modifie["ecart_type"] = (df_GBD_modifie["upper_bound"] - df_GBD_modifie["lower_bound"]) / (2 * 1.96)
df_GBD_modifie["var"] = df_GBD_modifie["ecart_type"]**2

df_GBD_modifie.head()

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound,ecart_type,var
0,71,Australia,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600,353.395281,1.248882e+05
1,71,Australia,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100,1211.078599,1.466711e+06
2,71,Australia,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387,39.459964,1.557089e+03
3,71,Australia,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975,0.192618,3.710169e-02
4,71,Australia,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420,15.432183,2.381523e+02


#### Aggrégation pour Malignant neoplasm (2)

In [ ]:
# filtration des causes qui relèvent de Malignant neoplasm 

cause_2 = [1012, 459, 489]

# pour chaque métrique, filtration & aggrégation & rajout du nombre de la métrique  

df_filtre_1 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 1)]
df_aggreg_1 = df_filtre_1.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_1 = df_aggreg_1.assign(
    metric_id = 1,
    metric_name = "Number"
)

df_filtre_2 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 2)]
df_aggreg_2 = df_filtre_2.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_2 = df_aggreg_2.assign(
    metric_id = 2,
    metric_name = "Percent"
)

df_filtre_3 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 3)]
df_aggreg_3 = df_filtre_3.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_3 = df_aggreg_3.assign(
    metric_id = 3,
    metric_name = "Rate"
)

df_aggreg_3.head()

In [ ]:
# Ajout des upper et lower bound pour chaque métrique (avec la métrique des variances)

df_aggreg_var_1 = df_filtre_1.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_1["ecart_type"] = np.sqrt(df_aggreg_var_1["var"])
df_aggreg_var_2 = df_filtre_2.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_2["ecart_type"] = np.sqrt(df_aggreg_var_2["var"])
df_aggreg_var_3 = df_filtre_3.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_3["ecart_type"] = np.sqrt(df_aggreg_var_3["var"])

L1 = df_aggreg_var_1["ecart_type"].tolist()
L1 = [1.96 * x for x in L1]
L2 = df_aggreg_var_2["ecart_type"].tolist()
L2 = [1.96 * x for x in L2]
L3 = df_aggreg_var_3["ecart_type"].tolist()
L3 = [1.96 * x for x in L3]

df_aggreg_1["lower_bound"] = df_aggreg_1["value"] - L1
df_aggreg_1["upper_bound"] = df_aggreg_1["value"] + L1
df_aggreg_2["lower_bound"] = df_aggreg_2["value"] - L2
df_aggreg_2["upper_bound"] = df_aggreg_2["value"] + L2
df_aggreg_3["lower_bound"] = df_aggreg_3["value"] - L3
df_aggreg_3["upper_bound"] = df_aggreg_3["value"] + L3

df_aggreg_3.head()

In [ ]:
# concaténation des trois dataframe

df_aggreg = pd.concat([df_aggreg_1, df_aggreg_2, df_aggreg_3], ignore_index=True)

df_aggreg.head()

In [ ]:
# rajout des autres colonnes et rangement 

df_aggreg = df_aggreg.assign(
    cause_id = 2,
    cause_name = "Malignant neoplasm",
)
df_aggreg = df_aggreg.sort_values(by=["location_name", "year","metric_id"]).reset_index(drop=True)


df_aggreg.head()

# Essai avec une fonction pour généraliser l'aggrégation des causes

Introduction des paramètres de fonction

In [47]:
# introduction des métriques

metric = ["Number", "Percent", "Rate"]

cause_2 = [1012, 459, 489] 

# introduction des nouvelles causes 

causes_generales = [["Malignant neoplasm", 2, [1012, 459, 489]], ["Motor vehicle accidents", 6, [689, 688]], ["Homicide", 9, [724, 945,854]]]

Fonction qui prend en argument un dataframe, la liste des causes associées à une cause générale et renvoie une liste de trois dataframe (un pour chaque métrique) qui contient le pays, l'année, les valeurs aggrégées pour cette cause générale et la métrique

In [48]:
def dataframe_par_metrique(df, cause, metric) :
    L = [] # initialisation de la liste
    for i in range(1,4) : # pour chaque métrique
        df_filtre_i = df[(df["cause_id"].isin(cause)) & (df["metric_id"] == i)] # filtration
        df_aggreg_i = df_filtre_i.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() # aggrégation
        df_aggreg_i = df_aggreg_i.assign(
            metric_id = i,
            metric_name = metric[i-1]
        ) # ajout du nom de la métrique  
        L.append(df_aggreg_i)
    return(L)

dataframe_par_metrique(df_GBD_modifie,cause_2, metric)

[      location_id location_name  year        value  metric_id metric_name
 0              47       Czechia  1980   943.567925          1      Number
 1              47       Czechia  1981   936.655643          1      Number
 2              47       Czechia  1982   932.900040          1      Number
 3              47       Czechia  1983   950.685882          1      Number
 4              47       Czechia  1984   986.545496          1      Number
 ...           ...           ...   ...          ...        ...         ...
 1667          155       Türkiye  2019  2349.845349          1      Number
 1668          155       Türkiye  2020  2448.668022          1      Number
 1669          155       Türkiye  2021  2509.954887          1      Number
 1670          155       Türkiye  2022  2695.978660          1      Number
 1671          155       Türkiye  2023  2835.548990          1      Number
 
 [1672 rows x 6 columns],
       location_id location_name  year     value  metric_id metric_name


Fonction qui prend en argument un dataframe, une liste de trois dataframe et la liste des causes associées à une cause générale, et renvoie la liste des trois dataframe en ayant ajouté le lower_bound et le higher_bound

In [49]:
def ajout_bounds(df, L, cause) :
    N = []
    for i in range(1,4) : 
        df_filtre_i = df[(df["cause_id"].isin(cause)) & (df["metric_id"] == i)] # filtration
        df_aggreg_var_i = df_filtre_i.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
        df_aggreg_var_i["ecart_type"] = np.sqrt(df_aggreg_var_i["var"])
        
        M = df_aggreg_var_i["ecart_type"].tolist()
        M = [1.96 * x for x in M]

        L[i-1]["lower_bound"] = L[i-1]["value"] - M
        L[i-1]["upper_bound"] = L[i-1]["value"] + M
        N.append(L[i-1])
    return(N)

ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2)

[      location_id location_name  year        value  metric_id metric_name  \
 0              47       Czechia  1980   943.567925          1      Number   
 1              47       Czechia  1981   936.655643          1      Number   
 2              47       Czechia  1982   932.900040          1      Number   
 3              47       Czechia  1983   950.685882          1      Number   
 4              47       Czechia  1984   986.545496          1      Number   
 ...           ...           ...   ...          ...        ...         ...   
 1667          155       Türkiye  2019  2349.845349          1      Number   
 1668          155       Türkiye  2020  2448.668022          1      Number   
 1669          155       Türkiye  2021  2509.954887          1      Number   
 1670          155       Türkiye  2022  2695.978660          1      Number   
 1671          155       Türkiye  2023  2835.548990          1      Number   
 
       lower_bound  upper_bound  
 0      878.273547  1008.862

Fonction qui prend en argument une liste de dataframe et renvoie leur concaténation

In [50]:
def concatenation(N) :
    df_concatene = pd.concat(N, ignore_index=True)
    return (df_concatene)

concatenation(ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2))


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172
...,...,...,...,...,...,...,...,...
5011,155,Türkiye,2019,2.830499,3,Rate,2.342761,3.318236
5012,155,Türkiye,2020,2.923271,3,Rate,2.407299,3.439243
5013,155,Türkiye,2021,2.970430,3,Rate,2.419421,3.521439
5014,155,Türkiye,2022,3.161625,3,Rate,2.584937,3.738313


Fonction qui prend en argument un dataframe, une cause générale (liste avec nom, numéro, et liste de numéro) et renvoie le même dataframe en ayant rajouté la cause

In [51]:
def ajout_cause (df, cause_generale) :
    df = df.assign(
    cause_id = cause_generale[1],
    cause_name = cause_generale[0],
    )
    return(df)

ajout_cause(concatenation(ajout_bounds(df_GBD_modifie,dataframe_par_metrique(df_GBD_modifie,cause_2, metric),cause_2)), causes_generales[0])


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound,cause_id,cause_name
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303,2,Malignant neoplasm
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099,2,Malignant neoplasm
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171,2,Malignant neoplasm
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024,2,Malignant neoplasm
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172,2,Malignant neoplasm
...,...,...,...,...,...,...,...,...,...,...
5011,155,Türkiye,2019,2.830499,3,Rate,2.342761,3.318236,2,Malignant neoplasm
5012,155,Türkiye,2020,2.923271,3,Rate,2.407299,3.439243,2,Malignant neoplasm
5013,155,Türkiye,2021,2.970430,3,Rate,2.419421,3.521439,2,Malignant neoplasm
5014,155,Türkiye,2022,3.161625,3,Rate,2.584937,3.738313,2,Malignant neoplasm


Fonction prend en argument un dataframe, une liste de causes générales, une métrique et renvoie le dataframe qui aggrège ce dernier par rapport aux causes générales

In [52]:
def aggrege(df, causes_generales, metric) :
    A = [] # future liste des data frame pour chaque cause générale
    for j in range(len(causes_generales)) :
        L = dataframe_par_metrique(df, causes_generales[j][2], metric) 
        N = ajout_bounds(df, L, causes_generales[j][2]) 
        dataframe = concatenation(N)
        A.append(ajout_cause (dataframe, causes_generales[j]))
    return(concatenation(A))

aggrege(df_GBD_modifie, causes_generales, metric)


,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound,cause_id,cause_name
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303,2,Malignant neoplasm
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099,2,Malignant neoplasm
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171,2,Malignant neoplasm
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024,2,Malignant neoplasm
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172,2,Malignant neoplasm
...,...,...,...,...,...,...,...,...,...,...
15043,155,Türkiye,2019,2.265427,3,Rate,1.808295,2.722558,9,Homicide
15044,155,Türkiye,2020,2.092415,3,Rate,1.588890,2.595940,9,Homicide
15045,155,Türkiye,2021,1.953624,3,Rate,1.371192,2.536055,9,Homicide
15046,155,Türkiye,2022,1.991516,3,Rate,1.355205,2.627828,9,Homicide


In [56]:
# on enlève ecart-type et variance

### Ajout des données qui n'ont pas eu besoin d'être aggrégées

In [ ]:
# création d'un data frame qui contient les valeurs qui n'ont pas du être agrégées

numero_non_agg = [294,491,521,322,718,696] # numeros des causes non aggrégées 
df_sans_modif = df_GBD_modifie[df_GBD_modifie["cause_id"].isin(numero_non_agg)]

# renommer les nom et numéro des causes

df_sans_modif.loc[df_sans_modif["cause_name"] == "All causes", "cause_id"] = 1

df_sans_modif.head()

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound,ecart_type,var
0,71,Australia,1,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600,353.395281,1.248882e+05
1,71,Australia,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100,1211.078599,1.466711e+06
2,71,Australia,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387,39.459964,1.557089e+03
5,71,Australia,322,Lower respiratory infections,1,Number,1980,1818.331630,1921.866841,1690.713494,58.967691,3.477189e+03
11,71,Australia,718,Self-harm,1,Number,1980,1746.605680,1853.616903,1649.878521,51.974077,2.701305e+03


In [ ]:
# réordonner

df_aggreg = df_aggreg.sort_values(by=["location_name", "year","metric_id"]).reset_index(drop=True)